# 05 — Domain presets showcase

The engine ships 6 domain presets — each a YAML file that tunes search bias, synthesize-prompt rules, and verification strictness for a specific field. This tutorial:

1. Loads all 6 presets and shows their differences.
2. Runs the **same question** through 3 different presets with a mocked LLM — observe how the synthesize prompt changes.
3. Walks through writing your own preset in ~10 lines of YAML.

**Time:** ~4 minutes. **Cost:** $0. **API key:** none.

## Step 1 — install

In [ ]:
%%capture
!git clone --depth 1 https://github.com/TheAiSingularity/agentic-research-engine-oss.git /content/engine-repo
!pip install -q langgraph openai rank_bm25 trafilatura pypdf pyyaml

import sys
sys.path.insert(0, '/content/engine-repo')

## Step 2 — enumerate the shipped presets

In [ ]:
from engine.core import domains

print('Presets shipped with the engine:')
for name in domains.list_names():
    print('  -', name)

In [ ]:
# Load and compare every preset side-by-side.
import textwrap

for name in domains.list_names():
    p = domains.load(name)
    print(f'━━━ {name} ━━━')
    print('  description:     ', textwrap.fill(p.description, width=70, subsequent_indent=' ' * 20)[:160])
    print('  min_verified:    ', p.min_verified_ratio)
    print('  top_k_evidence:  ', p.top_k_evidence)
    print('  searxng_cats:    ', p.searxng_categories)
    print('  seed_queries:    ', p.seed_queries[:2], '...' if len(p.seed_queries) > 2 else '')
    print('  tools_enabled:   ', p.tools_enabled)
    print('  prompt_extra[:60]:', (p.synthesize_prompt_extra or '').strip()[:60])
    print()

## Step 3 — same question, 3 different presets

Mock the LLM so we can inspect what question each preset's `_apply_domain_preset` hands to the pipeline.

In [ ]:
from types import SimpleNamespace
from unittest import mock
import engine.core.models as _models
import engine.core.pipeline as _pipeline
import requests

captured_prompts = []

def _resp(t): return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=t))])
def canned(*a, **kw):
    p = kw.get('messages', [{}])[0].get('content', '')
    captured_prompts.append(p)  # collect every prompt the pipeline sends
    if 'Classify' in p:                      return _resp('synthesis')
    if 'step-level verifier' in p:            return _resp('VERDICT: accept\nFEEDBACK:')
    if 'Break this research question' in p:   return _resp('sub one\nsub two\nsub three')
    if 'Summarize these sources' in p:         return _resp('Generic summary [1] [2].')
    if 'Compress each numbered chunk' in p:   return _resp('[1] c1\n[2] c2\n[3] c3')
    if 'Answer the question using ONLY' in p: return _resp('Answer tailored to the domain [1].')
    if 'List each standalone factual' in p:   return _resp('CLAIM: Something verifiable\nVERIFIED: yes')
    return _resp('(stub)')

cli = mock.MagicMock(); cli.chat.completions.create.side_effect = canned
_models.OpenAI = mock.MagicMock(return_value=cli)

def fake_get(url, params=None, timeout=None):
    r = mock.MagicMock(); r.status_code = 200; r.raise_for_status = mock.MagicMock()
    r.json = lambda: {'results': [{'url': 'https://a', 'title': 'A', 'content': 'A snip'}, {'url': 'https://b', 'title': 'B', 'content': 'B snip'}]}
    return r
requests.get = fake_get

from core.rag import HybridRetriever
_orig = HybridRetriever.__init__
def _p(self, *a, **kw):
    _orig(self, *a, **kw)
    self.embedder = lambda batch: [[float(len(s)), float(len(s.split()))] for s in batch]
HybridRetriever.__init__ = _p

_pipeline.ENABLE_STREAM = False
_pipeline.ENABLE_FETCH  = False
print('Mocks ready.')

In [ ]:
from engine.interfaces.common import run_query

QUESTION = 'What is the evidence on omega-3 supplementation for cardiovascular health?'

for domain_name in ('general', 'medical', 'papers'):
    captured_prompts.clear()
    rr = run_query(QUESTION, domain=domain_name)

    print(f'═══ DOMAIN: {domain_name} ═══')
    # The synthesize prompt is the one containing the domain-preset suffix.
    synthesize_prompts = [p for p in captured_prompts if 'Answer the question using ONLY' in p]
    if synthesize_prompts:
        snippet = synthesize_prompts[0]
        # Pull out the question-tail (where the prompt_suffix lands).
        tail = snippet.split('\n\nEvidence:')[0][-600:]
        print('synthesize prompt tail (last ~600 chars, includes domain-preset rules):')
        print(tail)
    print()
    print(f'--- answer for domain={domain_name} ---')
    print(rr.answer)
    print()

## Step 4 — write your own preset

Say you want a `legal_cases` preset that biases toward case-law databases and requires verbose citations.

In [ ]:
import pathlib

PRESET_YAML = '''name: legal_cases
description: >
  US legal case research. Biases search toward court opinions and legal
  databases. Requires court + year + docket on every case citation.

searxng_categories: []
seed_queries:
  - "site:supreme.justia.com"
  - "site:courtlistener.com"
  - "site:casetext.com"

synthesize_prompt_extra: |
  DOMAIN: legal_cases. Extra rules:
    - Cite the court, year, and docket number when available.
    - Distinguish binding precedent from persuasive authority.
    - This is research, not legal advice.

min_verified_ratio: 0.85
top_k_evidence: 8
tools_enabled: []
'''

# Drop it at the usual location — next CLI invocation picks it up.
preset_path = pathlib.Path('/content/engine-repo/engine/domains/legal_cases.yaml')
preset_path.write_text(PRESET_YAML)

# Verify the loader accepts it.
preset = domains.load('legal_cases')
print('Loaded:', preset.name)
print('min_verified_ratio:', preset.min_verified_ratio)
print('seed_queries:', preset.seed_queries)
print('prompt_extra preview:')
print(preset.synthesize_prompt_extra[:200])

In [ ]:
# Use the new preset.
rr = run_query('What did Marbury v. Madison establish?', domain='legal_cases')
print(rr.answer)

## What to try next

- Write a preset for your specialty (crypto research? scientific policy? forensic archaeology?) and open a PR — see `docs/domains.md` for the contributor lane.
- Pair a domain with `LOCAL_CORPUS_PATH` (Tutorial 03) for the strongest grounding — e.g. legal research on a pre-curated case-law dump.
- Tune `min_verified_ratio` per domain — high-stakes domains (medical, legal) warrant ≥0.80; exploratory domains can go lower.

That's all 5 tutorials. For deeper reading:
- [`engine/README.md`](https://github.com/TheAiSingularity/agentic-research-engine-oss/blob/main/engine/README.md) — quickstart
- [`docs/architecture.md`](https://github.com/TheAiSingularity/agentic-research-engine-oss/blob/main/docs/architecture.md) — deep spec
- [`docs/domains.md`](https://github.com/TheAiSingularity/agentic-research-engine-oss/blob/main/docs/domains.md) — preset schema
- [`CONTRIBUTING.md`](https://github.com/TheAiSingularity/agentic-research-engine-oss/blob/main/CONTRIBUTING.md) — good-first-issues